# Tutorial for basics usage of the JPL Small Body Identification Tool

## Intro
The notebook demonstrates the basic usage of `JPLSBID` class which provides a wrapper and front end for the JPL [Small-Body Identification Tool](https://ssd.jpl.nasa.gov/tools/sb_ident.html). This tool  This tool allows you to search for and retrieve a list of small-bodies (asteroids and/or comets) which are likely contained in the specified fiel of view on the specified date/time. 

## Tutorial

### Import needed things


In [ ]:
import sys
from astropy import units as u
from astropy.time import Time
from astropy.coordinates import SkyCoord

In [ ]:
print(sys.path)

In [ ]:
from solsys_code.views import JPLSBId

### Create a helper function to return the location of the Rubin DDFs (without needing the whole of `rubin_sim`)

In [ ]:
def special_locations(skycoords=False):
    """Special locations for Rubin. Adapted and condensed into one routine from
    the originals in `rubin_scheduler.utils`
    Parameters
    ----------
    skycoords : `bool`
        Return locations as astropy.SkyCoords. If False, returns
        as a tuple of floats. Default False.
    """

    # The DDF locations here are from Neil Brandt's white paper
    # submitted in response to the 2018 Call for White Papers on observing
    # strategy
    # Document-30468 -- AGN-DDF-WP-02.pdf
    # The locations are chosen based on existing multi-wavelength
    # coverage, plus an offset to avoid the bright star Mira near XMM-LSS

    result = {}
    result['ELAISS1'] = SkyCoord('00:37:48 −44:01:30', unit=(u.hourangle, u.deg), frame='icrs')
    result['XMM_LSS'] = SkyCoord('02:22:18  −04:49:00', unit=(u.hourangle, u.deg), frame='icrs')
    result['ECDFS'] = SkyCoord('03:31:55  −28:07:00', unit=(u.hourangle, u.deg), frame='icrs')
    result['COSMOS'] = SkyCoord('10:00:26  +02:14:01', unit=(u.hourangle, u.deg), frame='icrs')
    result['EDFS_a'] = SkyCoord(ra=58.90 * u.deg, dec=-49.32 * u.deg, frame='icrs')
    result['EDFS_b'] = SkyCoord(ra=63.60 * u.deg, dec=-47.60 * u.deg, frame='icrs')
    result['Roman_bulge_location'] = SkyCoord(268.708 * u.deg, -28.975 * u.deg, frame='icrs')

    if not skycoords:
        # replace SkyCoord with ra/deg tuple in degrees
        result = dict([(key, (result[key].ra.deg, result[key].dec.deg)) for key in result])

    return result

### Declare a time and points of interest 

This particular example is the (rough) start of alerts generation from the DDF fields


In [ ]:
obs_time = Time('2026-02-25T04:30:00', scale='utc')
ddfs = special_locations(True)

### Declare a JPLSBId object 
We declare a JPLSBId object and we can just use the defaults to initialize with the LSSTCam field of view (converted to a half-width). We do set the `two_pass` property to `True` to get more accurate positions

In [ ]:
# Declare a JPLSBId object and initialize with the ComCam field of view (converted to a half-width)
rubin = JPLSBId()
rubin.two_pass = True
print(f'MPC Code: {rubin.mpc_code} FOV= {rubin.fov_ra_hwidth:.3f} x {rubin.fov_dec_hwidth:.3f}')

### Query the JPL Small Body ID tool

This can take 30-60s. The defaults (`raw_response`=True, `verbose`=True) will show details of the query and return the raw (JSON) response but in this case we set `raw_response=False` to turn the results into a `Table`


In [ ]:
results_tables = {}
for field in ['COSMOS', 'EDFS_a', 'EDFS_b']:
    print(f'Querying {field} for {time}')
    results_table = rubin.query_center(time, ddfs[field], raw_response=False)
    results_tables[field] = results_table
print(results_tables.keys())

### Looking at the summary

The basic summary of the query is in the dictionary of results under the `summary` key (the RA and Dec are '-'-separated and minus signs in the Dec are convered to `M`)


In [ ]:
results_table = results_table['COSMOS']
for key, value in results_table['summary'].items():
    print(f'{key:<14}: {value}')

### Examining results
The per-object data is a list of fields (also a list) under the `data_first_pass` key with the fields ordered as described in `fields_first`

In [ ]:
print('\n'.join(results['fields_first']))
print()
print(results['data_first_pass'][0])

### Turning results into a Table
If instead, we set `raw_response` to False, the list of objects will be turned into a `QTable` with the position as a `SkyCoord` and `Quantity` columns

In [ ]:
table = rubin_comcam.query_center(obs_time, ecdfs, raw_response=False)
print(table)
print(table.colnames)